In [ ]:
import os, sys, subprocess, shlex, pathlib\n\nSCRIPT = \"TrainSemiDETR_SingleClass_STATIC_Ucloud.py\"\n\ndef _detect_runtime_profile() -> str:\n    forced = os.environ.get(\"RFDETR_RUNTIME_PROFILE\", \"auto\").strip().lower()\n    if forced in {\"ucloud\", \"local\"}:\n        return forced\n    if os.name != \"nt\" and pathlib.Path(\"/work\").exists():\n        has_member = any(pathlib.Path(\"/work\").glob(\"Member Files:*\"))\n        has_hash = any(pathlib.Path(\"/work\").glob(\"*#*\"))\n        if has_member or has_hash:\n            return \"ucloud\"\n    return \"local\"\n\nRUNTIME_PROFILE = _detect_runtime_profile()\nos.environ[\"RFDETR_RUNTIME_PROFILE\"] = RUNTIME_PROFILE\nif RUNTIME_PROFILE == \"ucloud\":\n    PROJECT_DIR = os.environ.get(\"PROJECT_DIR\", \"/work/projects/myproj/SOLO_SemiSupervised_RFDETR/\")\nelse:\n    PROJECT_DIR = os.environ.get(\"PROJECT_DIR\", str(pathlib.Path.cwd() / \"SOLO_SemiSupervised_RFDETR\"))\n\nPROJECT_PATH = pathlib.Path(PROJECT_DIR).resolve()\nREPO_ROOT = PROJECT_PATH.parent\n\nDATASET_ROOT = pathlib.Path(os.environ.get(\"STAT_DATASETS_ROOT\", str(REPO_ROOT / \"SOLO_Supervised_RFDETR\" / \"Stat_Dataset\"))).resolve()\ndef _latest_dataset_dir(root: pathlib.Path, token: str) -> str:\n    cands = sorted([p for p in root.glob(f\"*{token}*\") if p.is_dir()])\n    return str(cands[-1]) if cands else \"\"\nos.environ[\"DATASET_EPI\"] = os.environ.get(\"DATASET_EPI\", \"\").strip() or _latest_dataset_dir(DATASET_ROOT, \"SquamousEpithelialCell_OVR\")\nos.environ[\"DATASET_LEUCO\"] = os.environ.get(\"DATASET_LEUCO\", \"\").strip() or _latest_dataset_dir(DATASET_ROOT, \"Leucocyte_OVR\")\n\nif RUNTIME_PROFILE == \"ucloud\":\n    user_base = os.environ.get(\"USER_BASE_DIR\", \"\").strip()\n    if not user_base:\n        work = pathlib.Path(\"/work\")\n        members = sorted(work.glob(\"Member Files:*\"))\n        hashed = sorted([p for p in work.glob(\"*#*\") if p.is_dir()])\n        user_base = members[0].name if members else (hashed[0].name if hashed else \"\")\n    if user_base:\n        os.environ.setdefault(\"IMAGES_FALLBACK_ROOT\", f\"/work/{user_base}/CellScanData/Zoom10x - Quality Assessment_Cleaned\")\n        os.environ.setdefault(\"SEMIDETR_OUTPUT_ROOT\", f\"/work/{user_base}/SemiDETR_OUTPUT\")\nelse:\n    os.environ.setdefault(\"IMAGES_FALLBACK_ROOT\", str(pathlib.Path(r\"D:\\\\PHD\\\\PhdData\\\\CellScanData\\\\Zoom10x - Quality Assessment_Cleaned\")))\n    os.environ.setdefault(\"SEMIDETR_OUTPUT_ROOT\", str(PROJECT_PATH / \"SemiDETR_OUTPUT_local\"))\n\nos.environ.setdefault(\"SEMIDETR_REPO_DIR\", str(REPO_ROOT / \"_external_SemiDETR_ref\"))\nos.environ.setdefault(\"SEMIDETR_BASE_CONFIG\", str(pathlib.Path(os.environ[\"SEMIDETR_REPO_DIR\"]) / \"configs\" / \"detr_ssod\" / \"detr_ssod_dino_detr_r50_coco_120k.py\"))\n\n# Experiment setup\nos.environ[\"SEMIDETR_TARGET\"] = \"epi\"\nos.environ[\"SEMIDETR_INIT_MODE\"] = \"scratch\"\nos.environ[\"SEMIDETR_TRAIN_FRACTIONS\"] = \"1.0,0.5\"\nos.environ[\"SEMIDETR_SEEDS\"] = \"42\"\nos.environ[\"SEMIDETR_FRACTION_SEEDS\"] = \"42\"\n\n# Semi-DETR protocol knobs\nos.environ.setdefault(\"SEMIDETR_MAX_ITERS\", \"120000\")\nos.environ.setdefault(\"SEMIDETR_EVAL_INTERVAL\", \"4000\")\nos.environ.setdefault(\"SEMIDETR_CKPT_INTERVAL\", \"4000\")\nos.environ.setdefault(\"SEMIDETR_SAMPLES_PER_GPU\", \"5\")\nos.environ.setdefault(\"SEMIDETR_WORKERS_PER_GPU\", \"5\")\nos.environ.setdefault(\"SEMIDETR_SAMPLE_RATIO\", \"1,4\")\nos.environ.setdefault(\"SEMIDETR_PSEUDO_SCORE_THRESH\", \"0.5\")\nos.environ.setdefault(\"SEMIDETR_UNSUP_WEIGHT\", \"4.0\")\nos.environ.setdefault(\"SEMIDETR_UNLABELED_IMAGES_PER_LABELED\", \"6\")\nos.environ.setdefault(\"SEMIDETR_UNLABELED_MAX_IMAGES\", \"0\")\n\n# Preflight\nfor k in [\"DATASET_EPI\", \"DATASET_LEUCO\", \"IMAGES_FALLBACK_ROOT\", \"SEMIDETR_REPO_DIR\", \"SEMIDETR_BASE_CONFIG\"]:\n    p = os.environ.get(k, \"\")\n    if not p:\n        raise ValueError(f\"{k} is empty\")\n    if not pathlib.Path(p).exists():\n        raise FileNotFoundError(f\"Configured path does not exist for {k}: {p}\")\n\nprint(\"RFDETR_RUNTIME_PROFILE =\", RUNTIME_PROFILE)\nprint(\"PROJECT_DIR =\", PROJECT_DIR)\nfor k in [\n    \"SEMIDETR_TARGET\", \"SEMIDETR_INIT_MODE\", \"SEMIDETR_TRAIN_FRACTIONS\",\n    \"SEMIDETR_SEEDS\", \"SEMIDETR_FRACTION_SEEDS\",\n    \"SEMIDETR_MAX_ITERS\", \"SEMIDETR_SAMPLE_RATIO\",\n    \"SEMIDETR_PSEUDO_SCORE_THRESH\", \"SEMIDETR_UNSUP_WEIGHT\",\n    \"SEMIDETR_REPO_DIR\", \"SEMIDETR_BASE_CONFIG\",\n    \"DATASET_EPI\", \"DATASET_LEUCO\", \"IMAGES_FALLBACK_ROOT\", \"SEMIDETR_OUTPUT_ROOT\",\n]:\n    print(f\"{k} = {os.environ.get(k)}\")\n\nwd = pathlib.Path(PROJECT_DIR).resolve()\nscript_path = wd / SCRIPT\nif not script_path.exists():\n    raise FileNotFoundError(f\"Missing script: {script_path}\")\n\ncmd = [sys.executable, \"-u\", str(script_path)]\nprint(\"[LAUNCH]\")\nprint(\" cwd:\", wd)\nprint(\" cmd:\", \" \\".join(shlex.quote(x) for x in cmd))\n\nproc = subprocess.Popen(cmd, cwd=str(wd), env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\nfor line in proc.stdout:\n    print(line, end=\"\")\nret = proc.wait()\nif ret != 0:\n    raise RuntimeError(f\"Run failed with exit code {ret}\")\n